# ARM Server vs x86 Lockbench Comparison

Comparing `arm_server_lockbench.csv` (ARM server) against `x86_fixed_occ.csv` (Intel x86).  
Shared thread range: 1–8. x86 also has 16 and 32 threads.

**Locks:** TAS, TTAS, CAS, Ticket, RW, OCC, RCU, Lock  
**Workloads:** mutex (write-only), rw (mixed reads), rcu (read-heavy)

In [ ]:
from pathlib import Path
from IPython.display import display

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

INT_COLS   = ['threads', 'cs_work', 'read_pct', 'total_ops', 'read_ops', 'write_ops']
FLOAT_COLS = ['ops_s', 'ns_op', 'fairness_min', 'fairness_max', 'fairness_ratio']

def resolve_csv(name):
    for base in [Path('.'), Path('results')]:
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError(name)

def load_csv(path, platform):
    df = pd.read_csv(path, sep=';', decimal=',')
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
    for c in FLOAT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['ops_m']    = df['ops_s'] / 1e6
    df['platform'] = platform
    return df

arm = load_csv(resolve_csv('arm_server_lockbench.csv'), 'arm')
x86 = load_csv(resolve_csv('x86_fixed_occ.csv'),        'x86')
all_df = pd.concat([arm, x86], ignore_index=True)

SHARED_THREADS = sorted(set(arm['threads'].dropna().astype(int).unique()) &
                        set(x86['threads'].dropna().astype(int).unique()))

def agg(df, group_cols):
    return (
        df.groupby(group_cols)
          .agg(
              runs        = ('ops_m', 'size'),
              ops_m_mean  = ('ops_m',           'mean'),
              ops_m_std   = ('ops_m',           'std'),
              ns_mean     = ('ns_op',           'mean'),
              fair_mean   = ('fairness_ratio',  'mean'),
              fair_std    = ('fairness_ratio',  'std'),
          )
          .reset_index()
    )

# ── Visual style ──────────────────────────────────────────────────────────────
LOCK_ORDER  = ['tas', 'ttas', 'cas', 'ticket', 'rw', 'occ', 'rcu', 'lock']
LOCK_LABELS = {'tas':'TAS','ttas':'TTAS','cas':'CAS','ticket':'Ticket',
               'rw':'RW','occ':'OCC','rcu':'RCU','lock':'Lock'}
LOCK_COLORS = {'tas':'#4e79a7','ttas':'#f28e2b','cas':'#e15759','ticket':'#76b7b2',
               'rw':'#59a14f','occ':'#af7aa1','rcu':'#ff9da7','lock':'#9c755f'}

PLAT_COLORS = {'arm': '#2ca02c', 'x86': '#1f77b4'}
PLAT_LABELS = {'arm': 'ARM Server', 'x86': 'Intel x86'}
PLAT_MARKER = {'arm': 's', 'x86': 'o'}

print('ARM threads  :', sorted(arm['threads'].dropna().astype(int).unique()))
print('x86 threads  :', sorted(x86['threads'].dropna().astype(int).unique()))
print('Shared threads:', SHARED_THREADS)
print()
print('ARM locks    :', sorted(arm['lock'].unique()))
print('x86 locks    :', sorted(x86['lock'].unique()))
print()
print('ARM workloads:', sorted(arm['workload'].unique()))
print('x86 workloads:', sorted(x86['workload'].unique()))

## 1. Mutex Workload — Throughput (shared thread range)

In [ ]:
mutex = all_df[
    (all_df['workload'] == 'mutex') &
    (all_df['cs_work'] == 0) &
    (all_df['threads'].isin(SHARED_THREADS))
].copy()

g_mutex = agg(mutex, ['platform', 'lock', 'threads']).sort_values(['lock', 'platform', 'threads'])

mutex_locks = [l for l in LOCK_ORDER if l in mutex['lock'].unique()]
n_locks = len(mutex_locks)
ncols = 3
nrows = (n_locks + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharey=False)
axes = axes.flatten()

for idx, lock in enumerate(mutex_locks):
    ax = axes[idx]
    for plat in ['arm', 'x86']:
        sub = g_mutex[(g_mutex['lock'] == lock) & (g_mutex['platform'] == plat)]
        if sub.empty:
            continue
        ax.errorbar(
            sub['threads'], sub['ops_m_mean'], yerr=sub['ops_m_std'],
            label=PLAT_LABELS[plat], color=PLAT_COLORS[plat],
            marker=PLAT_MARKER[plat], linewidth=1.8, capsize=3,
        )
    ax.set_title(f'Mutex — {LOCK_LABELS[lock]}')
    ax.set_xlabel('Threads')
    ax.set_ylabel('Throughput (M ops/s)')
    ax.set_xticks(SHARED_THREADS)
    ax.legend(fontsize=9)

for ax in axes[n_locks:]:
    ax.set_visible(False)

fig.suptitle('ARM Server vs Intel x86: Mutex Throughput per Lock', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 2. Mutex Workload — Throughput Ratio (x86 / ARM)

In [ ]:
pivot = (
    g_mutex
      .pivot_table(index=['lock', 'threads'], columns='platform', values='ops_m_mean')
      .reset_index()
)
pivot['x86_over_arm'] = (pivot['x86'] / pivot['arm']).round(3)
pivot['arm_over_x86'] = (pivot['arm'] / pivot['x86']).round(3)

ratio_table = pivot.pivot(index='threads', columns='lock', values='x86_over_arm')
ratio_table.columns = [LOCK_LABELS.get(c, c) for c in ratio_table.columns]
print('x86 / ARM throughput ratio (>1 means x86 is faster):')
display(ratio_table.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(ratio_table.values.astype(float), aspect='auto', cmap='RdYlGn',
               vmin=0.3, vmax=3.0)
plt.colorbar(im, ax=ax, label='x86 / ARM ratio')

ax.set_xticks(range(len(ratio_table.columns)))
ax.set_xticklabels(ratio_table.columns, rotation=30, ha='right')
ax.set_yticks(range(len(ratio_table.index)))
ax.set_yticklabels(ratio_table.index)
ax.set_xlabel('Lock')
ax.set_ylabel('Threads')
ax.set_title('x86 / ARM Throughput Ratio — Mutex Workload\n(green > 1 = x86 faster, red < 1 = ARM faster)')

for i in range(len(ratio_table.index)):
    for j in range(len(ratio_table.columns)):
        v = ratio_table.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                    color='black')

fig.tight_layout()
plt.show()

## 3. Mutex Workload — Fairness Comparison

In [ ]:
g_fair = g_mutex[g_mutex['threads'] > 1]

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharey=True)
axes = axes.flatten()

for idx, lock in enumerate(mutex_locks):
    ax = axes[idx]
    for plat in ['arm', 'x86']:
        sub = g_fair[(g_fair['lock'] == lock) & (g_fair['platform'] == plat)]
        if sub.empty:
            continue
        ax.errorbar(
            sub['threads'], sub['fair_mean'], yerr=sub['fair_std'],
            label=PLAT_LABELS[plat], color=PLAT_COLORS[plat],
            marker=PLAT_MARKER[plat], linewidth=1.8, capsize=3,
        )
    ax.set_title(f'Fairness — {LOCK_LABELS[lock]}')
    ax.set_xlabel('Threads')
    ax.set_ylabel('Fairness ratio (min/max)')
    ax.set_xticks([t for t in SHARED_THREADS if t > 1])
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)

for ax in axes[n_locks:]:
    ax.set_visible(False)

fig.suptitle('ARM Server vs Intel x86: Mutex Fairness per Lock', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 4. RW Workload — OCC vs RW lock, 0 / 50 / 100 % reads

In [ ]:
rw_df = all_df[
    (all_df['workload'] == 'rw') &
    (all_df['lock'].isin(['rw', 'occ'])) &
    (all_df['threads'].isin(SHARED_THREADS))
].copy()

g_rw = agg(rw_df, ['platform', 'lock', 'read_pct', 'threads'])

READ_PCTS = [0, 50, 100]
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=False)

for col, read_pct in enumerate(READ_PCTS):
    for row, lock in enumerate(['rw', 'occ']):
        ax = axes[row, col]
        for plat in ['arm', 'x86']:
            sub = g_rw[
                (g_rw['platform'] == plat) &
                (g_rw['lock'] == lock) &
                (g_rw['read_pct'] == read_pct)
            ]
            if sub.empty:
                continue
            ax.errorbar(
                sub['threads'], sub['ops_m_mean'], yerr=sub['ops_m_std'],
                label=PLAT_LABELS[plat], color=PLAT_COLORS[plat],
                marker=PLAT_MARKER[plat], linewidth=1.8, capsize=3,
            )
        ax.set_title(f'{LOCK_LABELS[lock]} — {read_pct}% reads')
        ax.set_xlabel('Threads')
        ax.set_ylabel('Throughput (M ops/s)')
        ax.set_xticks(SHARED_THREADS)
        ax.legend(fontsize=9)

fig.suptitle('ARM Server vs Intel x86: RW Workload Throughput', fontsize=13)
fig.tight_layout()
plt.show()

# OCC / RW ratio per platform at 100% reads
print('\nOCC / RW throughput ratio at 100% reads:')
for plat in ['arm', 'x86']:
    sub100 = g_rw[(g_rw['read_pct'] == 100) & (g_rw['platform'] == plat)]
    ratio = (
        sub100.pivot(index='threads', columns='lock', values='ops_m_mean')
              .assign(occ_over_rw=lambda d: d['occ'] / d['rw'])
              .round(3)
    )
    print(f'  {PLAT_LABELS[plat]}')
    display(ratio)

## 5. RCU Workload — 0 / 50 / 100 % reads

In [ ]:
rcu_df = all_df[
    (all_df['workload'] == 'rcu') &
    (all_df['lock'] == 'rcu') &
    (all_df['threads'].isin(SHARED_THREADS))
].copy()

g_rcu = agg(rcu_df, ['platform', 'read_pct', 'threads'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, read_pct in zip(axes, READ_PCTS):
    for plat in ['arm', 'x86']:
        sub = g_rcu[(g_rcu['platform'] == plat) & (g_rcu['read_pct'] == read_pct)]
        if sub.empty:
            continue
        ax.errorbar(
            sub['threads'], sub['ops_m_mean'], yerr=sub['ops_m_std'],
            label=PLAT_LABELS[plat], color=PLAT_COLORS[plat],
            marker=PLAT_MARKER[plat], linewidth=1.8, capsize=3,
        )
    ax.set_title(f'RCU — {read_pct}% reads')
    ax.set_xlabel('Threads')
    ax.set_ylabel('Throughput (M ops/s)')
    ax.set_xticks(SHARED_THREADS)
    ax.set_yscale('log')
    ax.legend(fontsize=9)

fig.suptitle('ARM Server vs Intel x86: RCU Workload Throughput (log scale)', fontsize=13)
fig.tight_layout()
plt.show()

# Cross-platform ratio for RCU
print('\nRCU throughput ratio x86/ARM by read_pct and threads:')
rcu_ratio = (
    g_rcu.pivot_table(index=['read_pct', 'threads'], columns='platform', values='ops_m_mean')
         .assign(x86_over_arm=lambda d: d['x86'] / d['arm'])
         .round(3)
)
display(rcu_ratio)

## 6. Latency (ns/op) — Single-threaded baseline comparison

In [ ]:
lat1 = all_df[
    (all_df['workload'] == 'mutex') &
    (all_df['cs_work'] == 0) &
    (all_df['threads'] == 1)
].copy()

g_lat1 = (
    lat1.groupby(['platform', 'lock'])
        .agg(ns_mean=('ns_op', 'mean'), ns_std=('ns_op', 'std'))
        .reset_index()
)

locks_present = [l for l in LOCK_ORDER if l in g_lat1['lock'].unique()]
x = np.arange(len(locks_present))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
for i, plat in enumerate(['arm', 'x86']):
    sub = g_lat1[g_lat1['platform'] == plat].set_index('lock').reindex(locks_present)
    ax.bar(x + i * width, sub['ns_mean'], width, yerr=sub['ns_std'],
           label=PLAT_LABELS[plat], color=PLAT_COLORS[plat],
           alpha=0.85, capsize=4, error_kw={'elinewidth': 1.2})

ax.set_xticks(x + width / 2)
ax.set_xticklabels([LOCK_LABELS[l] for l in locks_present])
ax.set_ylabel('Latency (ns/op)')
ax.set_title('Single-Thread Lock Latency: ARM Server vs Intel x86')
ax.legend()
fig.tight_layout()
plt.show()

# Table
lat_table = (
    g_lat1.pivot(index='lock', columns='platform', values='ns_mean')
          .reindex(locks_present)
          .assign(x86_over_arm=lambda d: d['x86'] / d['arm'])
          .round(3)
)
lat_table.index = [LOCK_LABELS.get(i, i) for i in lat_table.index]
lat_table.columns.name = None
print('Single-thread latency (ns/op):')
display(lat_table)

## 7. Scaling behaviour — throughput vs threads (all locks, both platforms)

In [ ]:
# Normalise throughput to each lock's single-thread value to compare scaling curves
g_scale = g_mutex.copy()
t1 = g_scale[g_scale['threads'] == 1][['platform', 'lock', 'ops_m_mean']].rename(
        columns={'ops_m_mean': 'ops_m_t1'})
g_scale = g_scale.merge(t1, on=['platform', 'lock'])
g_scale['scaled'] = g_scale['ops_m_mean'] / g_scale['ops_m_t1']

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharey=False)
axes = axes.flatten()

for idx, lock in enumerate(mutex_locks):
    ax = axes[idx]
    for plat in ['arm', 'x86']:
        sub = g_scale[(g_scale['lock'] == lock) & (g_scale['platform'] == plat)]
        if sub.empty:
            continue
        ax.plot(sub['threads'], sub['scaled'],
                label=PLAT_LABELS[plat], color=PLAT_COLORS[plat],
                marker=PLAT_MARKER[plat], linewidth=1.8)
    # ideal linear scaling reference
    ax.plot(SHARED_THREADS, SHARED_THREADS, 'k--', linewidth=0.8, label='Linear ideal')
    ax.set_title(f'Scaling — {LOCK_LABELS[lock]}')
    ax.set_xlabel('Threads')
    ax.set_ylabel('Normalised throughput (t=1 → 1.0)')
    ax.set_xticks(SHARED_THREADS)
    ax.legend(fontsize=8)

for ax in axes[n_locks:]:
    ax.set_visible(False)

fig.suptitle('ARM Server vs Intel x86: Mutex Lock Scaling (normalised to 1-thread)', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 8. x86-only: High-thread scaling (16 and 32 threads)

In [ ]:
x86_mutex = x86[
    (x86['workload'] == 'mutex') &
    (x86['cs_work'] == 0)
].copy()

g_x86_mutex = agg(x86_mutex, ['lock', 'threads']).sort_values(['lock', 'threads'])

ALL_X86_THREADS = sorted(x86_mutex['threads'].dropna().astype(int).unique())

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharey=False)
axes = axes.flatten()

for idx, lock in enumerate(mutex_locks):
    ax = axes[idx]
    sub = g_x86_mutex[g_x86_mutex['lock'] == lock]
    if sub.empty:
        continue
    ax.errorbar(
        sub['threads'], sub['ops_m_mean'], yerr=sub['ops_m_std'],
        color=PLAT_COLORS['x86'], marker='o', linewidth=1.8, capsize=3,
    )
    ax.set_title(f'x86 — {LOCK_LABELS[lock]}')
    ax.set_xlabel('Threads')
    ax.set_ylabel('Throughput (M ops/s)')
    ax.set_xticks(ALL_X86_THREADS)
    ax.tick_params(axis='x', labelrotation=45)

for ax in axes[n_locks:]:
    ax.set_visible(False)

fig.suptitle('Intel x86: Mutex Throughput up to 32 Threads', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()